<a href="https://colab.research.google.com/github/Chaile-Mendez/tp-analisis-labo/blob/Mateo/notebooks/TP_LABO_PEGUNTAS_MATEO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Redes sociales
## ¿Cual es el impacto de las redes sociales en la calidad de vida de los estudiantes?

**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión · Trabajo práctico grupal**

Este notebook responde dos preguntas sobre el impacto de las redes sociales en estudiantes:

- ¿Hay **redes sociales** que impactan más que otras?
- ¿Existe una **brecha de género** en ese impacto?

### 🤖 Uso de IA

La herramienta de IA utilizada fue **Claude** (Anthropic), que se empleó para **ayudar y apoyar en la escritura del código en las diferentes celdas del notebook** (carga y exploración de datos, transformaciones y gráficos).

# 3.1 | Carga y exploración inicial

Importamos todas las librerías que vamos a utiliza: `pandas` para las tablas, `numpy` para cuentas, `matplotlib` y `seaborn` para los gráficos y `sklearn` para la regresión lineal.

In [14]:
import numpy as np                       # números y cuentas
import pandas as pd                      # tablas (DataFrame)
import matplotlib.pyplot as plt          # gráficos
import seaborn as sns                    # gráficos
from sklearn.linear_model import LinearRegression   # regresión lineal

In [15]:
# Cargamos directamente el dataset elegido desde nuestro repositorio.

url = "https://raw.githubusercontent.com/Chaile-Mendez/tp-analisis-labo/Facu/data/raw/dataset_final"

df = pd.read_csv(url)

### A) Primeras y últimas filas

Primero miramos y exploramos el dataset. Con `head()` vemos las primeras cinco filas para verificar que el archivo se leyó bien y ver el formato de cada columna.

In [16]:
df.head()

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Relationship_Status
0,788,22,Male,Graduate,Canada,4.6,Instagram,No,6.7,7.0,NaN
1,703,24,Male,Undergraduate,USA,4.7,YouTube,No,7.0,6.2,NaN
2,610,24,Male,Graduate,India,2.7,Instagram,No,8.6,7.8,NaN
3,699,22,Female,Graduate,USA,6.5,YouTube,Yes,5.4,5.0,NaN
4,302,21,Male,High School,Australia,6.8,YouTube,Yes,5.7,5.0,NaN


Con `tail()` vemos las últimas cinco filas

In [17]:
df.tail()

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Relationship_Status
1738,701,20,Female,Undergraduate,Italy,4.7,TikTok,No,7.2,7.0,In Relationship
1739,702,23,Male,Graduate,Russia,6.8,Instagram,Yes,5.9,4.0,Single
1740,703,21,Female,Undergraduate,China,5.6,WeChat,Yes,6.7,6.0,In Relationship
1741,704,24,Male,Graduate,Japan,4.3,Twitter,No,7.5,8.0,Single
1742,705,19,Female,Undergraduate,Poland,6.2,Facebook,Yes,6.3,5.0,Single


### B) Estructura

Vemos las dimensiones (filas y columnas) y el tipo de dato de cada columna con `dtypes`.

In [18]:
print("Dimensiones (filas, columnas):", df.shape)
print("\nTipos de dato:")
print(df.dtypes)

Dimensiones (filas, columnas): (1743, 11)

Tipos de dato:
Student_ID                        int64
Age                               int64
Gender                           object
Academic_Level                   object
Country                          object
Avg_Daily_Usage_Hours           float64
Most_Used_Platform               object
Affects_Academic_Performance     object
Sleep_Hours_Per_Night           float64
Mental_Health_Score             float64
Relationship_Status              object
dtype: object


Ahora hacemos una exploracion un poco mas profunda para ver lo siguiente: cuántos países distintos hay, rango de edad, y cómo se reparten los estudiantes por género y por plataforma más usada.

In [19]:
print("Países distintos:", df["Country"].nunique())
print("Rango de edad   :", df["Age"].min(), "a", df["Age"].max())
print("\nEstudiantes por género:")
print(df["Gender"].value_counts())
print("\nEstudiantes por plataforma más usada:")
print(df["Most_Used_Platform"].value_counts())

Países distintos: 110
Rango de edad   : 18 a 24

Estudiantes por género:
Gender
Male      883
Female    860
Name: count, dtype: int64

Estudiantes por plataforma más usada:
Most_Used_Platform
Instagram    550
TikTok       350
Facebook     295
WhatsApp     108
Twitter      103
LinkedIn      87
YouTube       75
Snapchat      73
WeChat        30
LINE          24
VKontakte     24
KakaoTalk     24
Name: count, dtype: int64


**Interpretacion:** El dataset tiene 1.743 filas (estudiantes) de 18 a 24 años de 110 países, la distribucion de genero es bastante pareja, aunque hay mas varones (883 varones y 860 mujeres). Hay **12 plataformas o redes**, muy desbalanceadas: Instagram (550) y TikTok (350) concentran más de la mitad de los casos, mientras que WeChat, LINE, KakaoTalk y VKontakte tienen 30 o menos observaciones cada una. Esto condiciona las comparaciones entre plataformas. Por ejemplo: los promedios de Instagram o TikTok, van a ser mas estables, mientras que los de WeChat o Lines, pueden moverse mucho por 1 o 2 estudiantes

### C) Calidad de los datos

Contamos los **nulos** por columna con `isnull().sum()`, las filas **duplicadas** con `duplicated().sum()`. Tambien utilizamos`describe()` para ver los estadisticos como maximos y minimos.
Ademas nos va a servir para revisar el identificador `Student_ID`. Si identifica a cada estudiante, no debería repetirse.

In [20]:
print("Nulos por columna:")
print(df.isnull().sum())                                 # isnull() marca cada celda vacía; sum() las cuenta por columna
print("\nFilas duplicadas:", df.duplicated().sum())       # duplicated() marca las filas idénticas a otra anterior

Nulos por columna:
Student_ID                         0
Age                                0
Gender                             0
Academic_Level                     0
Country                            0
Avg_Daily_Usage_Hours              0
Most_Used_Platform                 0
Affects_Academic_Performance       0
Sleep_Hours_Per_Night              0
Mental_Health_Score                0
Relationship_Status             1038
dtype: int64

Filas duplicadas: 0


In [21]:
df.describe().round(2)

,Student_ID,Age,Avg_Daily_Usage_Hours,Sleep_Hours_Per_Night,Mental_Health_Score
count,1743.00,1743.00,1743.00,1743.00,1743.00
mean,381.69,20.73,4.98,6.78,6.21
std,228.75,1.51,1.40,1.16,1.15
min,1.00,18.00,1.50,3.80,4.00
25%,188.50,19.00,4.00,5.90,5.00
50%,376.00,21.00,4.80,6.80,6.00
75%,562.00,22.00,5.90,7.60,7.00
max,1000.00,24.00,8.50,9.60,9.00


 Vemos que el `describe()` muestra un ID máximo de 1.000 con 1.743 filas, sospechamos que hay números repetidos, así que los contamos.

In [22]:
print("Student_ID repetidos:", df["Student_ID"].duplicated().sum(), "de", len(df), "filas")

Student_ID repetidos: 942 de 1743 filas


`Relationship_Status` tiene muchísimos nulos y `duplicated()` no encuentra nada, pero los `Student_ID` **se repiten** muchas veces. Sospechamos que las filas repetidas son el mismo estudiante cargado dos veces, y que el campo `Relationship_Status`  hace que parezcan filas distintas. Vamos a comprobarlo mediante un ejemplo: miramos un ID repetido y repetimos la búsqueda de duplicados **sin esa columna**.

In [23]:
# Las filas con Student_ID = 2
display(df[df["Student_ID"] == 2])

# Duplicados ignorando Relationship_Status
sin_relacion = df.drop(columns="Relationship_Status")
print("Duplicados ignorando Relationship_Status:", sin_relacion.duplicated().sum())

,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Relationship_Status
8,2,23,Female,Undergraduate,USA,2.7,YouTube,No,7.8,7.7,NaN
334,2,22,Male,Graduate,India,2.1,Twitter,No,7.5,8.0,NaN
1039,2,22,Male,Graduate,India,2.1,Twitter,No,7.5,8.0,Single


Duplicados ignorando Relationship_Status: 705


Como pudimos observar con el ejemplo, hay **705 filas repetidas**: son el mismo estudiante cargado dos veces, una con `Relationship_Status` vacío y otra con ese dato completo. (En el ejemplo del ID 2, las filas de India son el mismo estudiante; la de USA es otra persona que comparte el número.) Si no se las elimina, cada uno de esos estudiantes se cuenta dos veces y se inflan las muestras o promedios

**Concluciones:**

- `Relationship_Status` tiene **1.038 nulos**; es la única columna con faltantes.
- Hay **705 filas duplicadas** que no se detectaban por culpa de esa columna.
- No identificamos outliers: uso diario entre 1,5 y 8,5 horas, sueño entre 3,8 y 9,6 horas y salud mental entre 4 y 9 (a mayor puntaje, mejor salud mental).